# Notebook 01 — Data Preprocessing
**Mental Health Assessment Using Machine Learning**

---

## Purpose

This notebook transforms the raw survey CSV into two clean, analysis-ready datasets used by all subsequent notebooks. It handles column standardisation, psychometric scale scoring, clinical severity classification, majority-vote label creation, exploratory data visualisation, and natural-language narrative generation.

---

## Scale Scoring Reference

| Scale | Items | Response | Reverse-scored | Total Range | Severity → Class |
|-------|-------|----------|----------------|-------------|------------------|
| PSS-10 (Perceived Stress) | PSS1–PSS10 | 0–4 | PSS5, PSS6, PSS7, PSS8 | 0–40 | ≤13 → Stable · 14–26 → Challenged · ≥27 → Critical |
| GAD-7 (Anxiety) | GAD1–GAD7 | 0–3 | None | 0–21 | ≤4 → Stable · 5–9 → Challenged · ≥10 → Critical |
| PHQ-9 (Depression) | PHQ1–PHQ9 | 0–3 | None | 0–27 | ≤4 → Stable · 5–9 → Challenged · ≥10 → Critical |

**Mental Health Status** is determined by majority vote across the three per-scale classes (Stress Level, Anxiety Level, Depression Level). When all three differ, PHQ (Depression Level) acts as tiebreaker.

---

## Cell Map

| Cell | Summary |
|------|---------|
| 1 | Imports, plot config, working directory fix |
| 2 | Load raw CSV and inspect |
| 3 | Rename columns and standardise Department |
| 4 | PSS mapping, reverse scoring, PSS Total |
| 5 | GAD mapping and GAD Total |
| 6 | PHQ mapping and PHQ Total |
| 7 | Clinical thresholds → Stress Level, Anxiety Level, Depression Level |
| 8 | Majority voting → Mental Health Status · remove duplicates · class distribution |
| 9 | Reorder to final schema · assert shape · save `mha_tabular_dataset.csv` |
| 10 | Generate and save 13 EDA figures |
| 11 | Generate NLP narratives · save `mha_text_dataset.csv` |

## Cell 1 — Imports, Plot Configuration, Working Directory

Imports all libraries needed for the entire notebook. Matplotlib DPI is set to 300 for publication-quality figure exports. Seaborn's Set2 palette is used throughout for colourblind-friendly plots.

In [1]:
from pathlib import Path
import os, warnings
warnings.filterwarnings('ignore')
from collections import Counter

_cwd = Path.cwd()
if _cwd.name == 'notebooks':
    os.chdir(_cwd.parent)
print(f'Working directory: {Path.cwd()}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
    'font.family'      : 'DejaVu Sans',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})
sns.set_palette('Set2')

print('\n✓ All imports successful')

Working directory: d:\Programming\Projects\Mental Health Assessment

✓ All imports successful


## Cell 2 — Load Raw CSV and Inspect

Loads the raw survey file and prints a full initial inspection: shape, data types, null counts per column, duplicate row count, value counts for each of the 7 demographic columns, and the unique response string formats for PSS, GAD, and PHQ items. This inspection confirms the data state before any transformation and verifies that all three scale response formats are consistent.

In [2]:
RAW_PATH = os.path.join('data', 'raw', 'mhp_dataset.csv')
df_raw   = pd.read_csv(RAW_PATH)

print(f'Shape  : {df_raw.shape}  (rows × cols)')
print(f'Rows   : {df_raw.shape[0]:,}')
print(f'Cols   : {df_raw.shape[1]}')
print()

print('--- DTYPES ---')
print(df_raw.dtypes.to_string())
print()

print('--- NULL COUNTS ---')
nulls = df_raw.isnull().sum()
print(f'Total nulls: {nulls.sum()}')
print(nulls[nulls > 0] if nulls.sum() > 0 else 'No nulls found.')
print()

print(f'--- DUPLICATES ---')
print(f'Duplicate rows: {df_raw.duplicated().sum()}')
print()

print('--- DEMOGRAPHIC VALUE COUNTS (raw column names) ---')
for col in df_raw.columns[:7]:
    print(f'\n{col}:')
    print(df_raw[col].value_counts().to_string())
print()

print('--- SCALE RESPONSE FORMATS ---')
print('PSS (sample PSS1):', sorted(df_raw.iloc[:, 7].dropna().unique()))
print('GAD (sample GAD1):', sorted(df_raw.iloc[:, 17].dropna().unique()))
print('PHQ (sample PHQ1):', sorted(df_raw.iloc[:, 24].dropna().unique()))

Shape  : (2028, 33)  (rows × cols)
Rows   : 2,028
Cols   : 33

--- DTYPES ---
1. Age                                                                                                                                                                   str
2. Gender                                                                                                                                                                str
3. University                                                                                                                                                            str
4. Department                                                                                                                                                            str
5. Academic Year                                                                                                                                                         str
6. Current CGPA                                          

## Cell 3 — Rename Columns and Standardise Department

Renames all 33 original columns to clean, short keys matching the final schema: demographic columns become `Age`, `Gender`, `University`, `Department`, `Year`, `CGPA`, `Scholarship`; scale item columns become `PSS1`–`PSS10`, `GAD1`–`GAD7`, `PHQ1`–`PHQ9`. An assertion verifies all 33 columns were renamed correctly.

The `Department` column is then mapped from 12 granular categories (e.g. `Engineering - CS / CSE / CSC / Similar to CS`) to 5 broad groups: `Engineering`, `Science`, `Business`, `Arts & Social Sciences`, and `Other`. A second assertion confirms no unmapped values remain.

In [3]:
RENAME_MAP = {
    # Demographics
    '1. Age'                                                                                                                                                                   : 'Age',
    '2. Gender'                                                                                                                                                                : 'Gender',
    '3. University'                                                                                                                                                            : 'University',
    '4. Department'                                                                                                                                                            : 'Department',
    '5. Academic Year'                                                                                                                                                         : 'Year',
    '6. Current CGPA'                                                                                                                                                          : 'CGPA',
    '7. Did you receive a waiver or scholarship at your university?'                                                                                                           : 'Scholarship',
    # PSS-10
    '1. In a semester, how often have you felt upset due to something that happened in your academic affairs? '                                                                : 'PSS1',
    '2. In a semester, how often you felt as if you were unable to control important things in your academic affairs?'                                                         : 'PSS2',
    '3. In a semester, how often you felt nervous and stressed because of academic pressure? '                                                                                 : 'PSS3',
    '4. In a semester, how often you felt as if you could not cope with all the mandatory academic activities? (e.g, assignments, quiz, exams) '                              : 'PSS4',
    '5. In a semester, how often you felt confident about your ability to handle your academic / university problems?'                                                         : 'PSS5',
    '6. In a semester, how often you felt as if things in your academic life is going on your way? '                                                                          : 'PSS6',
    '7. In a semester, how often are you able to control irritations in your academic / university affairs? '                                                                  : 'PSS7',
    '8. In a semester, how often you felt as if your academic performance was on top?'                                                                                         : 'PSS8',
    '9. In a semester, how often you got angered due to bad performance or low grades that is beyond your control? '                                                           : 'PSS9',
    '10. In a semester, how often you felt as if academic difficulties are piling up so high that you could not overcome them? '                                               : 'PSS10',
    # GAD-7
    '1. In a semester, how often you felt nervous, anxious or on edge due to academic pressure? '                                                                             : 'GAD1',
    '2. In a semester, how often have you been unable to stop worrying about your academic affairs? '                                                                         : 'GAD2',
    '3. In a semester, how often have you had trouble relaxing due to academic pressure? '                                                                                    : 'GAD3',
    '4. In a semester, how often have you been easily annoyed or irritated because of academic pressure?'                                                                     : 'GAD4',
    '5. In a semester, how often have you worried too much about academic affairs? '                                                                                          : 'GAD5',
    '6. In a semester, how often have you been so restless due to academic pressure that it is hard to sit still?'                                                            : 'GAD6',
    '7. In a semester, how often have you felt afraid, as if something awful might happen?'                                                                                   : 'GAD7',
    # PHQ-9
    '1. In a semester, how often have you had little interest or pleasure in doing things?'                                                                                   : 'PHQ1',
    '2. In a semester, how often have you been feeling down, depressed or hopeless?'                                                                                         : 'PHQ2',
    '3. In a semester, how often have you had trouble falling or staying asleep, or sleeping too much? '                                                                      : 'PHQ3',
    '4. In a semester, how often have you been feeling tired or having little energy? '                                                                                       : 'PHQ4',
    '5. In a semester, how often have you had poor appetite or overeating? '                                                                                                  : 'PHQ5',
    '6. In a semester, how often have you been feeling bad about yourself - or that you are a failure or have let yourself or your family down? '                             : 'PHQ6',
    '7. In a semester, how often have you been having trouble concentrating on things, such as reading the books or watching television? '                                    : 'PHQ7',
    "8. In a semester, how often have you moved or spoke too slowly for other people to notice? Or you've been moving a lot more than usual because you've been restless? " : 'PHQ8',
    '9. In a semester, how often have you had thoughts that you would be better off dead, or of hurting yourself? '                                                           : 'PHQ9',
}

df = df_raw.rename(columns=RENAME_MAP)

EXPECTED = (
    ['Age','Gender','University','Department','Year','CGPA','Scholarship'] +
    [f'PSS{i}' for i in range(1, 11)] +
    [f'GAD{i}' for i in range(1,  8)] +
    [f'PHQ{i}' for i in range(1, 10)]
)
assert list(df.columns) == EXPECTED, 'Column rename mismatch — check RENAME_MAP'
print(f'✓ All 33 columns renamed')

DEPT_MAP = {
    'Engineering - CS / CSE / CSC / Similar to CS'        : 'Engineering',
    'Engineering - EEE/ ECE / Similar to EEE'             : 'Engineering',
    'Engineering - Mechanical Engineering / Similar to ME' : 'Engineering',
    'Engineering - Civil Engineering / Similar to CE'     : 'Engineering',
    'Engineering - Other'                                 : 'Engineering',
    'Biological Sciences'                                 : 'Science',
    'Environmental and Life Sciences'                     : 'Science',
    'Pharmacy and Public Health'                          : 'Science',
    'Business and Entrepreneurship Studies'               : 'Business',
    'Liberal Arts and Social Sciences'                    : 'Arts & Social Sciences',
    'Law and Human Rights'                                : 'Other',
    'Other'                                               : 'Other',
}
df['Department'] = df['Department'].map(DEPT_MAP)
assert df['Department'].isnull().sum() == 0, 'Unmapped Department values'
print('✓ Department mapped to 5 groups')
print()
print('Department distribution:')
print(df['Department'].value_counts().to_string())

✓ All 33 columns renamed
✓ Department mapped to 5 groups

Department distribution:
Department
Engineering               1658
Business                   145
Science                    144
Other                       80
Arts & Social Sciences       1


## Cell 4 — PSS Numeric Mapping, Reverse Scoring, and PSS Total

Extracts the leading digit from each PSS response string (e.g. `'3 - Fairly Often'` → `3`) and casts it to integer. This approach is robust to any minor variation in response text. PSS items **PSS5, PSS6, PSS7, and PSS8** are positively worded (confidence, things going well, ability to control, performance on top) and must be reverse-scored by computing `4 − score`. The PSS Total is then the sum of all 10 items (after reverse scoring), ranging from 0 to 40.

In [4]:
PSS_COLS    = [f'PSS{i}' for i in range(1, 11)]
PSS_REVERSE = ['PSS5', 'PSS6', 'PSS7', 'PSS8']

for col in PSS_COLS:
    df[col] = df[col].astype(str).str[0].astype(int)

for col in PSS_REVERSE:
    df[col] = 4 - df[col]

df['PSS Total'] = df[PSS_COLS].sum(axis=1)

print(f'PSS items mapped  : {PSS_COLS}')
print(f'Reverse-scored    : {PSS_REVERSE}')
print(f'PSS Total range   : {df["PSS Total"].min()} – {df["PSS Total"].max()}  (expected 0–40)')
print()
print(df['PSS Total'].describe().round(2))

PSS items mapped  : ['PSS1', 'PSS2', 'PSS3', 'PSS4', 'PSS5', 'PSS6', 'PSS7', 'PSS8', 'PSS9', 'PSS10']
Reverse-scored    : ['PSS5', 'PSS6', 'PSS7', 'PSS8']
PSS Total range   : 0 – 40  (expected 0–40)

count    2028.00
mean       23.00
std         6.76
min         0.00
25%        19.00
50%        22.00
75%        27.00
max        40.00
Name: PSS Total, dtype: float64


## Cell 5 — GAD Numeric Mapping and GAD Total

Extracts the leading digit from each GAD-7 response string (range 0–3; no reverse scoring required). The GAD Total is the sum of all 7 items, ranging from 0 to 21. Clinical thresholds applied in Cell 7: ≤4 = Stable (Minimal), 5–9 = Challenged (Mild), ≥10 = Critical (Moderate to Severe).

In [5]:
GAD_COLS = [f'GAD{i}' for i in range(1, 8)]

for col in GAD_COLS:
    df[col] = df[col].astype(str).str[0].astype(int)

df['GAD Total'] = df[GAD_COLS].sum(axis=1)

print(f'GAD Total range : {df["GAD Total"].min()} – {df["GAD Total"].max()}  (expected 0–21)')
print()
print(df['GAD Total'].describe().round(2))

GAD Total range : 0 – 21  (expected 0–21)

count    2028.00
mean       12.35
std         5.49
min         0.00
25%         8.00
50%        13.00
75%        17.00
max        21.00
Name: GAD Total, dtype: float64


## Cell 6 — PHQ Numeric Mapping and PHQ Total

Extracts the leading digit from each PHQ-9 response string (range 0–3; no reverse scoring required). The PHQ Total is the sum of all 9 items, ranging from 0 to 27. Clinical thresholds applied in Cell 7: ≤4 = Stable (None–Minimal), 5–9 = Challenged (Mild), ≥10 = Critical (Moderate to Severe).

In [6]:
PHQ_COLS = [f'PHQ{i}' for i in range(1, 10)]

for col in PHQ_COLS:
    df[col] = df[col].astype(str).str[0].astype(int)

df['PHQ Total'] = df[PHQ_COLS].sum(axis=1)

print(f'PHQ Total range : {df["PHQ Total"].min()} – {df["PHQ Total"].max()}  (expected 0–27)')
print()
print(df['PHQ Total'].describe().round(2))

PHQ Total range : 0 – 27  (expected 0–27)

count    2028.00
mean       14.43
std         6.67
min         0.00
25%         9.00
50%        15.00
75%        19.00
max        27.00
Name: PHQ Total, dtype: float64


## Cell 7 — Clinical Thresholds → Stress Level, Anxiety Level, Depression Level

Applies established clinical scoring thresholds to convert each scale total into a three-class label. These per-scale labels — `Stress Level`, `Anxiety Level`, and `Depression Level` — each take one of three values: `Stable`, `Challenged`, or `Critical`. They feed into the majority vote in Cell 8 and are stored as named columns in the final dataset.

| Scale | Stable | Challenged | Critical |
|-------|--------|------------|----------|
| PSS Total (0–40) | ≤ 13 (Low Stress) | 14–26 (Moderate) | ≥ 27 (High) |
| GAD Total (0–21) | ≤ 4 (Minimal) | 5–9 (Mild) | ≥ 10 (Moderate–Severe) |
| PHQ Total (0–27) | ≤ 4 (None–Minimal) | 5–9 (Mild) | ≥ 10 (Moderate–Severe) |

In [7]:
def pss_level(score):
    if score <= 13:   return 'Stable'
    elif score <= 26: return 'Challenged'
    else:             return 'Critical'

def gad_level(score):
    if score <= 4:    return 'Stable'
    elif score <= 9:  return 'Challenged'
    else:             return 'Critical'

def phq_level(score):
    if score <= 4:    return 'Stable'
    elif score <= 9:  return 'Challenged'
    else:             return 'Critical'

df['Stress Level']     = df['PSS Total'].apply(pss_level)
df['Anxiety Level']    = df['GAD Total'].apply(gad_level)
df['Depression Level'] = df['PHQ Total'].apply(phq_level)

for col in ['Stress Level', 'Anxiety Level', 'Depression Level']:
    print(f'{col}:')
    print(df[col].value_counts().to_string())
    print()

Stress Level:
Stress Level
Challenged    1348
Critical       565
Stable         115

Anxiety Level:
Anxiety Level
Critical      1364
Challenged     505
Stable         159

Depression Level:
Depression Level
Critical      1473
Challenged     414
Stable         141



## Cell 8 — Majority Voting → Mental Health Status · Remove Duplicates

Determines each student's final `Mental Health Status` by majority vote across the three per-scale labels (`Stress Level`, `Anxiety Level`, `Depression Level`). With three voters and three classes, a 2-to-1 majority always produces a clear winner. When all three differ (a 3-way split with one of each class), the **Depression Level (PHQ)** acts as the tiebreaker — PHQ-9 is the most widely validated instrument for overall functioning impairment and is therefore given precedence.

In [8]:
def majority_vote(row):
    votes   = [row['Stress Level'], row['Anxiety Level'], row['Depression Level']]
    counts  = Counter(votes)
    max_cnt = max(counts.values())
    winners = [k for k, v in counts.items() if v == max_cnt]
    return winners[0] if len(winners) == 1 else row['Depression Level']

df['Mental Health Status'] = df.apply(majority_vote, axis=1)

before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Removed {before - len(df)} duplicate rows')
print(f'Shape after deduplication: {df.shape}')
print()
print('Mental Health Status distribution:')
vc  = df['Mental Health Status'].value_counts()
pct = df['Mental Health Status'].value_counts(normalize=True).mul(100).round(1)
for label in ['Stable', 'Challenged', 'Critical']:
    print(f'  {label:<12}: {vc[label]:>5}  ({pct[label]}%)')

Removed 6 duplicate rows
Shape after deduplication: (2022, 40)

Mental Health Status distribution:
  Stable      :   123  (6.1%)
  Challenged  :   607  (30.0%)
  Critical    :  1292  (63.9%)


## Cell 9 — Reorder Columns, Assert Shape, Save Tabular Dataset

Reorders all columns to the final 40-column schema specified in the project plan. An assertion verifies the shape is exactly (2022, 40) before saving. The dataset is written to `data/processed/mha_tabular_dataset.csv` without an index column.

In [9]:
FINAL_COLS = (
    ['Age', 'Gender', 'University', 'Department', 'Year', 'CGPA', 'Scholarship'] +
    [f'PSS{i}' for i in range(1, 11)] + ['PSS Total', 'Stress Level'] +
    [f'GAD{i}' for i in range(1,  8)] + ['GAD Total', 'Anxiety Level'] +
    [f'PHQ{i}' for i in range(1, 10)] + ['PHQ Total', 'Depression Level'] +
    ['Mental Health Status']
)

df = df[FINAL_COLS]
assert df.shape == (2022, 40), f'Expected (2022, 40), got {df.shape}'

os.makedirs(os.path.join('data', 'processed'), exist_ok=True)
TABULAR_PATH = os.path.join('data', 'processed', 'mha_tabular_dataset.csv')
df.to_csv(TABULAR_PATH, index=False)

print(f'✓ Saved : {TABULAR_PATH}')
print(f'  Shape : {df.shape}')
print(f'  Columns ({len(df.columns)}): {list(df.columns)}')

✓ Saved : data\processed\mha_tabular_dataset.csv
  Shape : (2022, 40)
  Columns (40): ['Age', 'Gender', 'University', 'Department', 'Year', 'CGPA', 'Scholarship', 'PSS1', 'PSS2', 'PSS3', 'PSS4', 'PSS5', 'PSS6', 'PSS7', 'PSS8', 'PSS9', 'PSS10', 'PSS Total', 'Stress Level', 'GAD1', 'GAD2', 'GAD3', 'GAD4', 'GAD5', 'GAD6', 'GAD7', 'GAD Total', 'Anxiety Level', 'PHQ1', 'PHQ2', 'PHQ3', 'PHQ4', 'PHQ5', 'PHQ6', 'PHQ7', 'PHQ8', 'PHQ9', 'PHQ Total', 'Depression Level', 'Mental Health Status']


## Cell 10 — Generate and Save 13 EDA Figures

Generates all 13 exploratory data analysis figures and saves them to `figures/Exploratory Data Analysis/` at 300 DPI. Each figure is closed immediately after saving to release memory.

| Figure | Description |
|--------|-------------|
| 01 | Gender distribution — pie chart |
| 02 | Mental Health Status distribution — bar chart with counts and percentages |
| 03 | Mental Health Status by Gender — grouped bar chart |
| 04 | Per-scale severity distributions — three pie charts (Stress Level, Anxiety Level, Depression Level) |
| 05 | Correlation of PSS Total, GAD Total, PHQ Total — heatmap |
| 06 | Inter-item correlation of PSS1–PSS10 — heatmap |
| 07 | Inter-item correlation of GAD1–GAD7 — heatmap |
| 08 | Inter-item correlation of PHQ1–PHQ9 — heatmap |
| 09 | Inter-item correlation of all 26 items combined — heatmap |
| 10 | Response distribution for PSS items — stacked bar chart |
| 11 | Response distribution for GAD items — stacked bar chart |
| 12 | Response distribution for PHQ items — stacked bar chart |
| 13 | Voting pattern analysis — how each scale's class vote aligns with the final Mental Health Status |

In [10]:
FIG_DIR = os.path.join('figures', 'Exploratory Data Analysis')
os.makedirs(FIG_DIR, exist_ok=True)

CLASS_ORDER  = ['Stable', 'Challenged', 'Critical']
CLASS_COLORS = {'Stable': '#2ecc71', 'Challenged': '#f39c12', 'Critical': '#e74c3c'}
PSS_COLS     = [f'PSS{i}' for i in range(1, 11)]
GAD_COLS     = [f'GAD{i}' for i in range(1,  8)]
PHQ_COLS     = [f'PHQ{i}' for i in range(1, 10)]

def save_fig(name):
    plt.savefig(os.path.join(FIG_DIR, name), dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  ✓ {name}')

print('Saving EDA figures...')

# 01 Gender distribution
fig, ax = plt.subplots(figsize=(7, 7))
gc = df['Gender'].value_counts()
ax.pie(gc.values, labels=gc.index, autopct='%1.1f%%', startangle=90,
       colors=sns.color_palette('Set2', len(gc)),
       wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax.set_title('Gender Distribution', fontsize=14, fontweight='bold', pad=15)
save_fig('01_gender_distribution.png')

# 02 Mental Health Status distribution
fig, ax = plt.subplots(figsize=(8, 5))
mhs = df['Mental Health Status'].value_counts().reindex(CLASS_ORDER)
bars = ax.bar(mhs.index, mhs.values,
              color=[CLASS_COLORS[c] for c in CLASS_ORDER], edgecolor='white')
for bar, val in zip(bars, mhs.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{val}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)
ax.set_title('Mental Health Status Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Mental Health Status'); ax.set_ylabel('Number of Students')
ax.set_ylim(0, mhs.max() * 1.2)
save_fig('02_mental_health_status_distribution.png')

# 03 Mental Health Status by Gender
fig, ax = plt.subplots(figsize=(10, 6))
ct = pd.crosstab(df['Gender'], df['Mental Health Status'])[CLASS_ORDER]
ct.plot(kind='bar', ax=ax, color=[CLASS_COLORS[c] for c in CLASS_ORDER], edgecolor='white')
ax.set_title('Mental Health Status by Gender', fontsize=14, fontweight='bold')
ax.set_xlabel('Gender'); ax.set_ylabel('Number of Students')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Mental Health Status')
save_fig('03_mental_health_status_by_gender.png')

# 04 Per-scale severity distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (col, title) in zip(axes, [
    ('Stress Level', 'PSS Stress Level'),
    ('Anxiety Level', 'GAD Anxiety Level'),
    ('Depression Level', 'PHQ Depression Level')
]):
    vc = df[col].value_counts().reindex(CLASS_ORDER)
    ax.pie(vc.values, labels=vc.index, autopct='%1.1f%%', startangle=90,
           colors=[CLASS_COLORS[c] for c in CLASS_ORDER],
           wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
    ax.set_title(title, fontsize=12, fontweight='bold')
fig.suptitle('Individual Scale Severity Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig('04_individual_severity_levels.png')

# 05 Correlation of raw scale totals
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(df[['PSS Total','GAD Total','PHQ Total']].corr(),
            annot=True, fmt='.3f', cmap='coolwarm', center=0, square=True,
            linewidths=0.5, ax=ax)
ax.set_title('Correlation — Scale Totals (PSS / GAD / PHQ)', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('05_correlation_raw_scores.png')

# 06 PSS item correlations
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[PSS_COLS].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.3, ax=ax, annot_kws={'size': 8})
ax.set_title('Inter-Item Correlation — PSS1–PSS10', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('06_correlation_pss_items.png')

# 07 GAD item correlations
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[GAD_COLS].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.3, ax=ax, annot_kws={'size': 9})
ax.set_title('Inter-Item Correlation — GAD1–GAD7', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('07_correlation_gad_items.png')

# 08 PHQ item correlations
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[PHQ_COLS].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.3, ax=ax, annot_kws={'size': 8})
ax.set_title('Inter-Item Correlation — PHQ1–PHQ9', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('08_correlation_phq_items.png')

# 09 All 26 items combined
ALL_ITEMS = PSS_COLS + GAD_COLS + PHQ_COLS
fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(df[ALL_ITEMS].corr(), cmap='coolwarm', center=0, square=True,
            linewidths=0.2, ax=ax, annot=False,
            xticklabels=ALL_ITEMS, yticklabels=ALL_ITEMS)
ax.set_title('Inter-Item Correlation — All 26 Scale Items', fontsize=14, fontweight='bold')
plt.xticks(fontsize=8, rotation=90); plt.yticks(fontsize=8, rotation=0)
plt.tight_layout()
save_fig('09_correlation_all_items_combined.png')

# 10-12 Response distributions (stacked bar)
def plot_response_dist(cols, freq_range, title, fname):
    counts = pd.DataFrame(
        {c: df[c].value_counts().reindex(range(freq_range)) for c in cols}
    ).fillna(0)
    ax = counts.T.plot(kind='bar', stacked=True, figsize=(13, 6),
                       color=sns.color_palette('Blues_d', freq_range), edgecolor='white')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Item'); ax.set_ylabel('Count')
    ax.set_xticklabels(cols, rotation=45, ha='right')
    ax.legend([str(i) for i in range(freq_range)], title='Score',
              bbox_to_anchor=(1.01, 1))
    plt.tight_layout()
    save_fig(fname)

plot_response_dist(PSS_COLS, 5, 'Response Distribution — PSS Stress Items (PSS1–PSS10)',
                   '10_responses_pss_stress.png')
plot_response_dist(GAD_COLS, 4, 'Response Distribution — GAD Anxiety Items (GAD1–GAD7)',
                   '11_responses_gad_anxiety.png')
plot_response_dist(PHQ_COLS, 4, 'Response Distribution — PHQ Depression Items (PHQ1–PHQ9)',
                   '12_responses_phq_depression.png')

# 13 Voting pattern analysis
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (vote_col, label) in zip(axes, [
    ('Stress Level',     'PSS Vote (Stress Level)'),
    ('Anxiety Level',    'GAD Vote (Anxiety Level)'),
    ('Depression Level', 'PHQ Vote (Depression Level)'),
]):
    ct = (pd.crosstab(df['Mental Health Status'], df[vote_col], normalize='index') * 100
          ).reindex(CLASS_ORDER)[CLASS_ORDER]
    ct.plot(kind='bar', ax=ax,
            color=[CLASS_COLORS[c] for c in CLASS_ORDER], edgecolor='white')
    ax.set_title(f'{label}\nby Final Mental Health Status', fontsize=10, fontweight='bold')
    ax.set_xlabel('Final Mental Health Status')
    ax.set_ylabel('Percentage (%)')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.legend(title='Vote', fontsize=8)
    ax.set_ylim(0, 108)
fig.suptitle('Voting Pattern Analysis: Per-Scale Class Votes vs Final Mental Health Status',
             fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('13_voting_pattern_analysis.png')

print(f'\n✓ All 13 EDA figures saved to: {FIG_DIR}')

Saving EDA figures...
  ✓ 01_gender_distribution.png
  ✓ 02_mental_health_status_distribution.png
  ✓ 03_mental_health_status_by_gender.png
  ✓ 04_individual_severity_levels.png
  ✓ 05_correlation_raw_scores.png
  ✓ 06_correlation_pss_items.png
  ✓ 07_correlation_gad_items.png
  ✓ 08_correlation_phq_items.png
  ✓ 09_correlation_all_items_combined.png
  ✓ 10_responses_pss_stress.png
  ✓ 11_responses_gad_anxiety.png
  ✓ 12_responses_phq_depression.png
  ✓ 13_voting_pattern_analysis.png

✓ All 13 EDA figures saved to: figures\Exploratory Data Analysis


## Cell 11 — Generate NLP Narratives and Save Text Dataset

Generates one natural-language paragraph per student (the `Student Information` column) that describes their demographics and all 26 scale item responses in plain English. Numeric scores are mapped back to frequency descriptors: PSS responses use `never / almost never / sometimes / fairly often / very often`; GAD and PHQ responses use `not at all / on several days / more than half the time / nearly every day`.

The paragraph includes: age group, gender, university, department, year of study, CGPA range, scholarship status, PSS Total with a plain-English sentence for each of the 10 items, GAD Total with 7 items, and PHQ Total with 9 items.

The text dataset is saved with two columns: `Student Information` (paragraph) and `Mental Health Status` (text label: Stable / Challenged / Critical).

In [11]:
PSS_FREQ    = {0: 'never', 1: 'almost never', 2: 'sometimes',
               3: 'fairly often', 4: 'very often'}
GADPHQ_FREQ = {0: 'not at all', 1: 'on several days',
               2: 'more than half the time', 3: 'nearly every day'}

PSS_DESC = {
    'PSS1' : 'felt upset due to unexpected academic events',
    'PSS2' : 'felt unable to control important academic matters',
    'PSS3' : 'felt nervous and stressed due to academic pressure',
    'PSS4' : 'felt unable to cope with mandatory academic activities',
    'PSS5' : 'felt confident in handling academic problems',
    'PSS6' : 'felt academic life was going their way',
    'PSS7' : 'felt able to control irritations in academic affairs',
    'PSS8' : 'felt their academic performance was on top',
    'PSS9' : 'felt angered by poor results beyond their control',
    'PSS10': 'felt academic difficulties were piling up uncontrollably',
}
GAD_DESC = {
    'GAD1': 'felt nervous, anxious, or on edge due to academic pressure',
    'GAD2': 'was unable to stop worrying about academic affairs',
    'GAD3': 'had trouble relaxing due to academic pressure',
    'GAD4': 'was easily annoyed or irritated due to academic pressure',
    'GAD5': 'worried excessively about academic matters',
    'GAD6': 'felt so restless due to academic pressure that sitting still was difficult',
    'GAD7': 'felt afraid as if something awful might happen',
}
PHQ_DESC = {
    'PHQ1': 'had little interest or pleasure in doing things',
    'PHQ2': 'felt down, depressed, or hopeless',
    'PHQ3': 'had trouble sleeping or slept too much',
    'PHQ4': 'felt tired or had little energy',
    'PHQ5': 'had poor appetite or was overeating',
    'PHQ6': 'felt bad about themselves or like a failure',
    'PHQ7': 'had trouble concentrating on things',
    'PHQ8': 'moved or spoke too slowly, or was unusually restless',
    'PHQ9': 'had thoughts of being better off dead or of self-harm',
}

PSS_COLS_NLP = [f'PSS{i}' for i in range(1, 11)]
GAD_COLS_NLP = [f'GAD{i}' for i in range(1,  8)]
PHQ_COLS_NLP = [f'PHQ{i}' for i in range(1, 10)]

def generate_narrative(row):
    scholarship = 'receives a scholarship' if row['Scholarship'] == 'Yes' \
                  else 'does not receive a scholarship'
    pss_text = '; '.join([
        f'{PSS_FREQ[row[c]]} {PSS_DESC[c]}' for c in PSS_COLS_NLP
    ])
    gad_text = '; '.join([
        f'{GADPHQ_FREQ[row[c]]} {GAD_DESC[c]}' for c in GAD_COLS_NLP
    ])
    phq_text = '; '.join([
        f'{GADPHQ_FREQ[row[c]]} {PHQ_DESC[c]}' for c in PHQ_COLS_NLP
    ])
    return (
        f"This student is a {row['Gender'].lower()} in the {row['Age']} age group, "
        f"studying {row['Department']} at {row['University']} in {row['Year']} "
        f"with a CGPA of {row['CGPA']} and {scholarship}. "
        f"Regarding perceived stress (PSS Total: {row['PSS Total']}/40): "
        f"during the semester they {pss_text}. "
        f"Regarding anxiety (GAD Total: {row['GAD Total']}/21): "
        f"they {gad_text}. "
        f"Regarding depression (PHQ Total: {row['PHQ Total']}/27): "
        f"they {phq_text}."
    )

df_text = pd.DataFrame({
    'Student Information'  : df.apply(generate_narrative, axis=1),
    'Mental Health Status' : df['Mental Health Status'],
})

TEXT_PATH = os.path.join('data', 'processed', 'mha_text_dataset.csv')
df_text.to_csv(TEXT_PATH, index=False)

print(f'✓ Saved : {TEXT_PATH}')
print(f'  Shape : {df_text.shape}  (expected (2022, 2))')
print(f'  Columns: {list(df_text.columns)}')
print(f'  Label distribution:')
print(df_text['Mental Health Status'].value_counts().to_string())
print()
print('Sample narrative (row 0):')
print(df_text['Student Information'].iloc[0])

✓ Saved : data\processed\mha_text_dataset.csv
  Shape : (2022, 2)  (expected (2022, 2))
  Columns: ['Student Information', 'Mental Health Status']
  Label distribution:
Mental Health Status
Critical      1292
Challenged     607
Stable         123

Sample narrative (row 0):
This student is a female in the 18-22 age group, studying Engineering at Independent University, Bangladesh (IUB) in Second Year or Equivalent with a CGPA of 2.50 - 2.99 and does not receive a scholarship. Regarding perceived stress (PSS Total: 29/40): during the semester they fairly often felt upset due to unexpected academic events; very often felt unable to control important academic matters; fairly often felt nervous and stressed due to academic pressure; sometimes felt unable to cope with mandatory academic activities; sometimes felt confident in handling academic problems; fairly often felt academic life was going their way; sometimes felt able to control irritations in academic affairs; sometimes felt their ac